# Introducción al Sistema de Análisis Hidro-Climático Global
El presente desarrollo constituye una herramienta de Hydro-Intelligence diseñada para la extracción y procesamiento automatizado de series temporales climáticas de alta resolución. El sistema integra la potencia de cómputo en la nube de Google Earth Engine (GEE) con el lenguaje de programación Python, permitiendo un balance hídrico expedito en cualquier coordenada geográfica del planeta.1. Fundamento Técnico y Escala de ObservaciónLa arquitectura del código se fundamenta en la coincidencia multiescalar. Se ha establecido un área circular de influencia (buffer) con un radio de 2,750 metros, lo que genera un diámetro de 5.5 km. Esta dimensión no es arbitraria; ha sido calibrada para coincidir exactamente con la resolución nominal de 0.05° del sensor CHIRPS V2.0.Esta relación Escala 1:1 garantiza que el promedio espacial de las variables extraídas sea estadísticamente representativo de la unidad mínima de información satelital disponible, eliminando sesgos de sub-muestreo o sobre-interpolación en terrenos de topografía compleja como la región andina.2. Fuentes de Datos y VariablesEl motor de cálculo realiza una consulta integrada a dos de los repositorios climáticos más robustos de la actualidad:Precipitación ($P$): Extraída de la serie diaria CHIRPS, que combina datos satelitales infrarrojos con registros de estaciones en tierra para una máxima precisión pluviométrica.Variables Termodinámicas: Datos de temperatura, humedad, viento y radiación neta provenientes del modelo ERA5-Land (ECMWF), con una resolución de 11.1 km, procesados mediante promedios ponderados dentro del buffer de estudio.3. Dinámica del Balance HídricoA partir de las variables térmicas y la oscilación de temperaturas extremas ($T_{max}$ y $T_{min}$), el sistema calcula la Evapotranspiración Potencial (ETP) mediante la metodología de Hargreaves-Samani (1985). El resultado final permite determinar el Balance Hídrico Neto ($P - ETP$), clasificando el estado hídrico del área en Superávit o Déficit, información crucial para la gestión de recursos hídricos y el cálculo de la huella hídrica verde en reservas naturales.

In [ ]:
import geemap
import ipywidgets as widgets
from IPython.display import display
import ee
import math
import matplotlib.pyplot as plt
import pandas as pd
from geopy.geocoders import Nominatim

# 1. INICIALIZACIÓN
PROJECT_ID = 'ee-rafabarcenas78'
try:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize()

geolocator = Nominatim(user_agent="geo_analytics_dr_prec")

# Controles
start_year = widgets.IntSlider(description='Año Inicio:', min=1993, max=2026, value=2020)
end_year = widgets.IntSlider(description='Año Fin:', min=1993, max=2026, value=2025)
controles = widgets.HBox([start_year, end_year])

Map_Final = geemap.Map()
Map_Final.setCenter(-75.48, 5.06, 12)
Map_Final.add_basemap('HYBRID')
output_panel = widgets.Output()

def generar_reporte_doctorado_preciso(lat, lon, s_year, e_year):
    with output_panel:
        output_panel.clear_output()

        # Geocodificación
        try:
            location = geolocator.reverse(f"{lat}, {lon}", timeout=10)
            address = location.raw.get('address', {})
            ciudad = address.get('city', address.get('town', address.get('village', 'N/A')))
            region = address.get('state', 'N/A')
            pais = address.get('country', 'N/A')
        except:
            ciudad, region, pais = "N/A", "N/A", "N/A"

        print(f"🌍 UBICACIÓN: {ciudad}, {region}, {pais}")
        print(f"📍 COORDENADAS: {lat:.4f}, {lon:.4f}")

        # --- AJUSTE DE PRECISIÓN: RADIO 2750m = DIÁMETRO 5500m (PÍXEL CHIRPS) ---
        for layer in Map_Final.layers:
            if layer.name == 'Área de Influencia (5.5km)':
                Map_Final.remove_layer(layer)

        punto = ee.Geometry.Point([lon, lat])
        buffer = punto.buffer(2750) # Radio de 2.75km para coincidir con el píxel de 5.5km
        Map_Final.addLayer(buffer, {'color': '#FF5722', 'fillColor': '#FF572233'}, 'Área de Influencia (5.5km)')

        rango = (f'{s_year}-01-01', f'{e_year}-12-31')
        num_anos = e_year - s_year + 1

        try:
            chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(*rango)
            era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").filterDate(*rango)

            data_list = []
            meses_nombres = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

            for m in range(1, 13):
                p_mes = chirps.filter(ee.Filter.calendarRange(m, m, 'month')).sum().divide(num_anos)
                p_val = p_mes.reduceRegion(ee.Reducer.mean(), buffer, 5000).get('precipitation').getInfo()

                era_mes = era5.filter(ee.Filter.calendarRange(m, m, 'month')).mean()
                bandas = ['temperature_2m', 'temperature_2m_max', 'temperature_2m_min',
                          'dewpoint_temperature_2m', 'u_component_of_wind_10m',
                          'v_component_of_wind_10m', 'surface_net_solar_radiation_sum']

                stats = era_mes.select(bandas).reduceRegion(ee.Reducer.mean(), buffer, 10000).getInfo()

                t_med = stats['temperature_2m'] - 273.15
                t_max = stats['temperature_2m_max'] - 273.15
                t_min = stats['temperature_2m_min'] - 273.15
                tdew = stats['dewpoint_temperature_2m'] - 273.15
                hr = 100 * (math.exp((17.625 * tdew) / (243.04 + tdew)) / math.exp((17.625 * t_med) / (243.04 + t_med)))
                u, v = stats['u_component_of_wind_10m'], stats['v_component_of_wind_10m']
                wind = math.sqrt(u**2 + v**2)
                rad_solar = stats['surface_net_solar_radiation_sum'] / 1000000
                etp_m = 0.0023 * 0.408 * 34.5 * (t_med + 17.8) * (math.sqrt(t_max - t_min)) * 30.4

                data_list.append({
                    'Mes': meses_nombres[m-1], 'P (mm)': round(p_val, 1), 'ETP (mm)': round(etp_m, 1),
                    'T_med (°C)': round(t_med, 2), 'Rad (MJ/m2/d)': round(rad_solar, 2),
                    'HR (%)': round(hr, 1), 'Viento (m/s)': round(wind, 2)
                })

            df = pd.DataFrame(data_list)

            # --- REPORTE CONSOLIDADO ---
            p_anual = df['P (mm)'].sum()
            etp_anual = df['ETP (mm)'].sum()
            hr_media = df['HR (%)'].mean()
            balance = p_anual - etp_anual

            print(f"\n✅ REPORTE CLIMÁTICO CONSOLIDADO ({s_year}-{e_year})\n" + "="*85)
            print(f"🌡️ TEMP MEDIA: {df['T_med (°C)'].mean():.2f} °C | 🌬️ VIENTO: {df['Viento (m/s)'].mean():.2f} m/s | 💧 HUMEDAD: {hr_media:.2f} %")
            print(f"🌧️ P. ANUAL: {p_anual:.1f} mm | 🔥 ETP ANUAL: {etp_anual:.1f} mm")
            print("-" * 85 + f"\n⚖️ BALANCE HÍDRICO NETO: {balance:.2f} mm/año | ESTADO: {'SUPERÁVIT' if balance > 0 else 'DÉFICIT'}")
            print("="*85)

            # --- GRÁFICA MULTIVARIABLE ---
            fig, ax1 = plt.subplots(figsize=(12, 5))
            ax1.bar(df['Mes'], df['P (mm)'], color='#3498db', alpha=0.4, label='Precipitación (mm)')
            ax1.plot(df['Mes'], df['ETP (mm)'], color='#e74c3c', marker='o', linewidth=2, label='ETP (mm)')
            ax1.set_ylabel('Agua (mm)', fontweight='bold')
            for i, v in enumerate(df['P (mm)']):
                ax1.text(i, v + 2, str(int(v)), ha='center', fontsize=9)

            ax2 = ax1.twinx()
            ax2.plot(df['Mes'], df['T_med (°C)'], color='#f39c12', linestyle='--', label='Temp (°C)')
            ax2.plot(df['Mes'], df['HR (%)'], color='#27ae60', linestyle=':', label='Hum (%)')
            ax2.set_ylabel('Temp / Humedad', fontweight='bold')
            plt.title(f'Balance Hídrico Mensual en {ciudad}, {pais}')
            fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
            plt.grid(axis='y', alpha=0.2)
            plt.show()

            # --- TABLA Y FUENTES ---
            display(df)
            print("\n📚 RECURSOS Y FUENTES DE DATOS UTILIZADOS:")
            print("-" * 85)
            print("1. Precipitación: CHIRPS Daily V2.0 (Resolución 5.5 km - 0.05°).")
            print("2. Variables Climáticas: ERA5-Land Monthly Aggregated (Resolución 11.1 km - 0.1°).")
            print("3. Metodología ETP: Ecuación de Hargreaves-Samani (1985).")
            print("4. Geoprocesamiento: Radio de 2,750 m (Diámetro 5.5 km) - Escala 1:1 con Píxel CHIRPS.")
            print("-" * 85)
            print("\n📝 PÁRRAFO METODOLÓGICO PARA TESIS:")
            print(f"Se estableció un área circular de influencia con un radio de 2,750 m, conformando un diámetro total de 5.5 km. "
                  f"Esta dimensión coincide exactamente con la resolución espacial nominal del producto CHIRPS V2.0 (0.05°), "
                  f"lo que permite una extracción de datos climatológicos sin sesgos de sub-muestreo espacial para la zona de {ciudad}, {pais}.")

        except Exception as e:
            print(f"❌ Error en el cálculo: {e}")

def handle_click(**kwargs):
    if kwargs.get('type') == 'click':
        lat, lon = kwargs.get('coordinates')
        generar_reporte_doctorado_preciso(lat, lon, start_year.value, end_year.value)

Map_Final.on_interaction(handle_click)
display(controles, Map_Final, output_panel)